In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_auc_score, roc_curve


In [2]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "tags").exists():
    PROJECT_DIR = PROJECT_DIR.parent

TAGS_DIR = PROJECT_DIR / "tags"
OUTPUT_DIR = PROJECT_DIR / "output" / "history_3d_simple_cnn_multilabel_1"

TRAIN_TAGS_CSV = TAGS_DIR / "train.csv"
VAL_TAGS_CSV = TAGS_DIR / "validation.csv"
TRAIN_PREDICTIONS_CSV = OUTPUT_DIR / "predict_train_3d_simple_cnn_multilabel_1.csv"
VAL_PREDICTIONS_CSV = OUTPUT_DIR / "predict_val_3d_simple_cnn_multilabel_1.csv"

score_column = "max_predict"


In [3]:
def load_split_predictions(tags_path, predictions_path, score_column="max_predict"):
    tags = pd.read_csv(tags_path)
    predictions = pd.read_csv(predictions_path)

    if "ICH" not in tags.columns:
        raise ValueError(f"Missing column in {tags_path}: ICH")
    if score_column not in predictions.columns:
        raise ValueError(f"Missing column in {predictions_path}: {score_column}")
    if len(tags) != len(predictions):
        raise ValueError(
            f"Different row counts: {tags_path} has {len(tags)}, "
            f"{predictions_path} has {len(predictions)}"
        )

    data = predictions.copy()
    data["ICH"] = tags["ICH"].astype(int).to_numpy()

    y_true = data["ICH"].to_numpy()
    y_score = pd.to_numeric(data[score_column], errors="raise").to_numpy()
    return data, y_true, y_score


train_predictions, train_y_true, train_y_score = load_split_predictions(
    tags_path=TRAIN_TAGS_CSV,
    predictions_path=TRAIN_PREDICTIONS_CSV,
    score_column=score_column,
)

val_predictions, val_y_true, val_y_score = load_split_predictions(
    tags_path=VAL_TAGS_CSV,
    predictions_path=VAL_PREDICTIONS_CSV,
    score_column=score_column,
)

print(f"Train: {len(train_predictions)} rows")
print(f"Validation: {len(val_predictions)} rows")
display(train_predictions.head())
display(val_predictions.head())


Train: 992 rows
Validation: 105 rows


,study_uid,prob_epidural hemorrhage,prob_subarachnoid hemorrhage,prob_subdural hemorrhage,prob_intracerebral hemorrhage,max_predict,ICH
0,1.2.643.5.1.13.13.12.2.77.8252.040410000607021...,0.592409,0.526734,0.342728,0.586889,0.592409,1
1,1.2.643.5.1.13.13.12.2.77.8252.111508140103091...,0.283878,0.254241,0.270596,0.417668,0.417668,0
2,1.2.643.5.1.13.13.12.2.77.8252.001001111115151...,0.028820,0.031267,0.145007,0.162118,0.162118,0
3,1.2.643.5.1.13.13.12.2.77.8252.061006030815120...,0.017301,0.019399,0.129944,0.124468,0.129944,0
4,1.2.643.5.1.13.13.12.2.77.8252.131312110705130...,0.019091,0.021290,0.136436,0.128989,0.136436,0


,study_uid,prob_epidural hemorrhage,prob_subarachnoid hemorrhage,prob_subdural hemorrhage,prob_intracerebral hemorrhage,max_predict,ICH
0,1.2.643.5.1.13.13.12.2.77.8252.030906081102080...,0.104748,0.100774,0.202535,0.279145,0.279145,0
1,1.2.643.5.1.13.13.12.2.77.8252.080702151114061...,0.039363,0.041297,0.156977,0.184854,0.184854,1
2,1.2.643.5.1.13.13.12.2.77.8252.120700110012120...,0.392009,0.349305,0.287485,0.489075,0.489075,1
3,1.2.643.5.1.13.13.12.2.77.8252.010207031003031...,0.030901,0.033179,0.147181,0.166282,0.166282,0
4,1.2.643.5.1.13.13.12.2.77.8252.111310111203000...,0.062043,0.062884,0.180673,0.224878,0.224878,0


In [4]:
def plot_preliminary_plots(y_true, y_score, dataset_name):
    fig_hist = px.histogram(
        x=y_score,
        color=y_true.astype(str),
        nbins=50,
        labels={"color": "True Labels", "x": "Score"},
        title=f"{dataset_name}: Histogram of Scores",
        width=800,
        height=500,
    )
    fig_hist.show()

    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    df_thresh = pd.DataFrame({"FPR": fpr, "TPR": tpr}, index=thresholds)

    fig_thresh = px.line(
        df_thresh,
        title=f"{dataset_name}: TPR and FPR at every threshold",
        width=800,
        height=500,
    )
    fig_thresh.update_yaxes(scaleanchor="x", scaleratio=1)
    fig_thresh.update_xaxes(range=[0, 1], constrain="domain")
    fig_thresh.show()


def plot_roc_curve(y_true, y_score, dataset_name):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    fig = px.area(
        x=fpr,
        y=tpr,
        title=f"{dataset_name}: ROC Curve (AUC={roc_auc:.3f})",
        labels={"x": "False Positive Rate", "y": "True Positive Rate"},
        width=800,
        height=500,
    )
    fig.add_shape(type="line", line=dict(dash="dash"), x0=0, x1=1, y0=0, y1=1)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()
    display(Markdown(f"**{dataset_name} ROC-AUC:** {roc_auc:.4f}"))
    return roc_auc


def plot_pr_curve(y_true, y_score, dataset_name):
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall, precision)
    average_precision = average_precision_score(y_true, y_score)

    fig = px.area(
        x=recall,
        y=precision,
        title=f"{dataset_name}: Precision-Recall Curve (AUC={pr_auc:.3f})",
        labels={"x": "Recall", "y": "Precision"},
        width=800,
        height=500,
    )
    fig.add_shape(type="line", line=dict(dash="dash"), x0=0, x1=1, y0=1, y1=0)
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.show()
    display(Markdown(f"**{dataset_name} PR-AUC:** {pr_auc:.4f}"))
    display(Markdown(f"**{dataset_name} Average Precision:** {average_precision:.4f}"))
    return pr_auc, average_precision


In [5]:
metric_rows = []

for y_true, y_score, name in [
    (train_y_true, train_y_score, "Train"),
    (val_y_true, val_y_score, "Validation"),
]:
    plot_preliminary_plots(y_true, y_score, name)
    roc_auc = plot_roc_curve(y_true, y_score, name)
    pr_auc, average_precision = plot_pr_curve(y_true, y_score, name)
    metric_rows.append(
        {
            "dataset": name,
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "average_precision": average_precision,
        }
    )

metrics = pd.DataFrame(metric_rows)
display(metrics)


**Train ROC-AUC:** 0.8511

**Train PR-AUC:** 0.8297

**Train Average Precision:** 0.8299

**Validation ROC-AUC:** 0.8237

**Validation PR-AUC:** 0.7648

**Validation Average Precision:** 0.7690

,dataset,roc_auc,pr_auc,average_precision
0,Train,0.851051,0.829669,0.829932
1,Validation,0.823704,0.764767,0.769029
